# ST-Reason · 04 Memory construction

Builds every memory format from the outputs of notebooks 02 and 03. Every level comes from the **same cleaned events and object states**, so comparing levels changes how the facts are organised, not which facts are present.

| Format | Content |
| --- | --- |
| **M1** prose | every event and state change as plain sentences in time order; no IDs, tables or relations |
| **M2** event list | `E1 [t=2.1-3.4s] person_1 picks up cup_3 (kitchen)` |
| **M3** events + states | M2 plus one line of state intervals per object |
| **M4** hierarchical | M3 plus a computed summary, 10 s segments, and computed temporal relations (BEFORE / AFTER / DURING / OVERLAPS, 0.5 s tolerance) and CO_OCCURS links |
| **CAP** (baseline) | the window captions, concatenated (MoReVQA-style caption-then-answer) |
| **VA** (baseline) | VideoAgent-style: segment captions plus an object table |
| **ORACLE_M1 … M4** | the same four formats built from STAR's ground truth (Charades actions and situation graphs), for the "perfect perception" runs |

It also creates the **STAR pause and sham versions** by shifting times in the original clips' outputs (and in the oracle ground truth), as described in notebook 03.

**Runtime:** CPU is enough (no GPU needed). About 10–20 minutes. It can be re-run any time: it rebuilds everything from the 02 and 03 results on Drive and uploads a new, verified zip; later notebooks always use the newest one.

| Output on Drive | Content |
| --- | --- |
| `outputs/memory/04_memories_<time>.zip` | one JSON per (format, video): the full text, the separate items (for retrieval in strategies C and D), and token counts |
| `outputs/memory/memory_stats.csv` | items, words and tokens per format and video |
| `outputs/memory/examples.txt` | every format for one example video, to read |

In [10]:
# --- Parameters ----------------------------------------------------------------------------------
SHARD, N_SHARDS = 0, 1        # not used here: one person runs this notebook
GROUPS = ['dev', 'star', 'main', 'intentqa']   # intentqa is used only if notebook 03 processed it

In [11]:
# --- 0. Bootstrap --------------------------------------------------------------------------
import os, glob
ST_NOTEBOOK = '04_memory_build'
ST_QUIET = True
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')  # before torch loads
if 'COLAB_RELEASE_TAG' in os.environ:
    from google.colab import drive
    drive.mount('/content/drive')
    SETUP_NB = '/content/drive/MyDrive/ST-Reason/notebooks/00_setup_and_utils.ipynb'
elif os.path.exists('/kaggle/working'):
    SETUP_NB = (glob.glob('/kaggle/input/**/00_setup_and_utils.ipynb', recursive=True) or ['00_setup_and_utils.ipynb'])[0]
else:
    SETUP_NB = os.environ.get('ST_SETUP_NB', '00_setup_and_utils.ipynb')
%run -i $SETUP_NB

MOCK = os.environ.get('ST_MOCK_MODELS') == '1'      # local testing only: fake models, no GPU
assert SHARD in range(N_SHARDS), 'SHARD must be 0 .. N_SHARDS-1'
# each member appends to their own manifest, so two people never write the same file at once
MANIFEST_PATH = P['runs'] / f'manifest_shard{SHARD}.csv'
# Log to the Colab disk; a copy goes to Drive with every upload. (Appending to a file on Drive line by line
# makes Drive store many duplicate copies of it.)
DRIVE_LOG_PATH = LOG_PATH
LOG_PATH = P['local'] / 'logs' / LOG_PATH.name
LOG_PATH.parent.mkdir(parents=True, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Runtime: colab | project folder: /content/drive/MyDrive/ST-Reason | local scratch: /content/st_local
config hash: 2a9e0e4f87


## 1. Load the CV results

All 02 and 03 packs on Drive are unpacked to the Colab disk (this happens in the next cell). Videos without 03 results are listed and skipped; run this notebook again once they are done.

In [12]:
# --- Shared CV helpers (this cell is identical in 02 and 03) -------------------------------------
import pandas as pd, numpy as np, cv2
from PIL import Image, ImageDraw, ImageFont

CVC = CFG['cv']
CVC.setdefault('openvocab_fps', 1.0)          # pilot: YOLOE at 1 fps is enough and 5x cheaper
CVC.setdefault('stitch_min_cos', 0.80)
CVC.setdefault('state_fill_gap_s', 1.0)
CVC.setdefault('state_min_s', 0.6)
CVC.setdefault('event_window_s', CVC['window_s'])   # pilot: non-overlapping 3 s windows
# Per-video files are written to the fast Colab disk (LOCAL_CV) and uploaded to Drive in batches, as a
# few zip "packs" placed directly in outputs/cv/. Each pack is read back from Drive after upload to prove
# it arrived. (Writing thousands of small files to Drive can silently lose them, and new sub-folders made
# from Colab inside a folder shared from someone else's Drive can vanish, so none are created.)
LOCAL_CV = P['local'] / 'cv_out'
PACKS = P['cv']
PACK_GLOB = '0[23]*_shard*.zip'
CVD = {k: LOCAL_CV / k for k in ['frames_index', 'tracks', 'states', 'events_raw', 'events', 'captions', 'frames']}
for _d in CVD.values():
    _d.mkdir(parents=True, exist_ok=True)
_PACKED = set()            # paths (relative to LOCAL_CV) already safely inside a pack on Drive

def restore_from_packs(prefixes=None):
    """Unpack every pack on Drive into LOCAL_CV (files already there are skipped).
    prefixes: only these top-level folders, e.g. ['states', 'tracks']."""
    n_new, bad = 0, []
    for z in sorted(PACKS.glob(PACK_GLOB), reverse=True):   # newest first: a redone video's newest copy wins
        try:
            with zipfile.ZipFile(z) as zf:
                for m in zf.namelist():
                    _PACKED.add(m)
                    if prefixes and m.split('/')[0] not in prefixes:
                        continue
                    out = LOCAL_CV / m
                    if not out.exists():
                        out.parent.mkdir(parents=True, exist_ok=True)
                        with zf.open(m) as s, open(out, 'wb') as d:
                            shutil.copyfileobj(s, d)
                        n_new += 1
        except (zipfile.BadZipFile, OSError) as e:
            bad.append(z.name)
    if bad:
        log(f'{len(bad)} damaged packs skipped (an upload was cut off; their videos are redone): {bad[:3]}', 'WARN')
    log(f'restored {n_new} files from {len(list(PACKS.glob(PACK_GLOB)))} packs on Drive')

class DriveSyncError(BaseException):
    """Stops the whole loop (run_resumable only skips ordinary errors), so no more work piles up unsaved."""

def sync_to_drive(subdirs, tag):
    """Upload every local file in `subdirs` that is not yet on Drive, as one new pack, and verify it."""
    new = []
    for sd in subdirs:
        for p in (LOCAL_CV / sd).rglob('*'):
            if p.is_file() and not p.name.endswith('.tmp'):
                rel = str(p.relative_to(LOCAL_CV))
                if rel not in _PACKED:
                    new.append(rel)
    if not new:
        return 0
    name = f'{tag}_shard{SHARD}_{dt.datetime.now():%Y%m%d-%H%M%S}_{len(new):05d}.zip'
    tmp = P['local_tmp'] / name
    with zipfile.ZipFile(tmp, 'w', zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
        for rel in new:
            zf.write(LOCAL_CV / rel, rel, compress_type=zipfile.ZIP_STORED if rel.endswith('.jpg') else zipfile.ZIP_DEFLATED)
    dest = PACKS / name
    for attempt in range(3):
        try:
            shutil.copyfile(tmp, dest)
            with zipfile.ZipFile(dest) as zf:                  # read back from Drive
                ok = len(zf.namelist()) == len(new) and zf.testzip() is None
            if ok:
                break
        except Exception as e:
            ok = False
            log(f'upload of {name} failed ({e.__class__.__name__}: {e}); retrying', 'WARN')
        time.sleep(5 * (attempt + 1))
    if not ok:
        raise DriveSyncError(f'Drive did not keep {name}. The results are still on this Colab disk: do NOT restart '
                           'the runtime. Check Drive space and that you can edit MyDrive/ST-Reason, then run the '
                           'sync cell again.')
    _PACKED.update(new)
    tmp.unlink(missing_ok=True)
    try:
        shutil.copyfile(LOG_PATH, DRIVE_LOG_PATH)          # one overwrite per upload, not one per line
    except Exception:
        pass
    log(f'uploaded {name}: {len(new)} files ({_gb(dest.stat().st_size) * 1024:.1f} MB), verified on Drive')
    return len(new)

restore_from_packs()

COCO = set(['person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck', 'boat', 'traffic light',
    'fire hydrant', 'stop sign', 'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow', 'elephant',
    'bear', 'zebra', 'giraffe', 'backpack', 'umbrella', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard',
    'sports ball', 'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 'tennis racket', 'bottle',
    'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich', 'orange', 'broccoli',
    'carrot', 'hot dog', 'pizza', 'donut', 'cake', 'chair', 'couch', 'potted plant', 'bed', 'dining table', 'toilet',
    'tv', 'laptop', 'mouse', 'remote', 'keyboard', 'cell phone', 'microwave', 'oven', 'toaster', 'sink',
    'refrigerator', 'book', 'clock', 'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush'])
# objects that other objects can rest "on"
SURFACES = {'dining table', 'table', 'desk', 'bed', 'couch', 'sofa', 'chair', 'bench', 'shelf', 'counter',
            'cabinet', 'toilet', 'sink', 'mat', 'rug', 'carpet', 'tray', 'stool', 'cushion', 'pillow'}

def save_compact(obj, path):
    """Atomic JSON write without indentation (CV files are large)."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + '.tmp')
    with open(tmp, 'w') as f:
        json.dump(obj, f, separators=(',', ':'), ensure_ascii=False, default=_json_default)
        f.flush(); os.fsync(f.fileno())
    os.replace(tmp, path)

def decode_frames(path, fps_out):
    """Yield (t_seconds, BGR frame) at about fps_out, decoding once and streaming.
    02 and 03 both use this, so every stage sees exactly the same timestamps."""
    cap = cv2.VideoCapture(str(path))
    src_fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    step, i, next_t = 1.0 / fps_out, 0, 0.0
    while True:
        ok, fr = cap.read()
        if not ok:
            break
        t = i / src_fps
        if t + 1e-6 >= next_t:
            yield round(t, 3), fr
            next_t += step
        i += 1
    cap.release()

def stratified_subset(df, n, seed):
    """Exactly n rows, allocated to question types in proportion to their counts (same as 01)."""
    n = min(n, len(df))
    counts = df.type.value_counts()
    alloc = (counts / counts.sum() * n).round().astype(int).clip(upper=counts)
    order, i = list(counts.index), 0
    while alloc.sum() != n:
        t = order[i % len(order)]
        if alloc.sum() < n and alloc[t] < counts[t]:
            alloc[t] += 1
        elif alloc.sum() > n and alloc[t] > 0:
            alloc[t] -= 1
        i += 1
    parts = [df[df.type == t].sample(n=int(k), random_state=seed) for t, k in alloc.items() if k > 0]
    return pd.concat(parts).sample(frac=1, random_state=seed).reset_index(drop=True)

def main_grid_eval():
    """The questions the main grid is actually run on. The pilot rule cut the grid to
    config data.main_grid_n (400). The subset keeps all 200 robustness questions and adds a
    stratified sample of the rest, so the robustness subset stays inside it. Frozen once saved."""
    path = P['subsets'] / 'main_grid_eval.csv'
    if path.exists():
        return pd.read_csv(path, dtype={'video_id': str})
    grid = pd.read_csv(P['subsets'] / 'main_grid.csv', dtype={'video_id': str})
    n = int(CFG['data'].get('main_grid_n') or (CFG.get('pilot_decisions') or {}).get('main_grid_n') or len(grid))
    if n >= len(grid):
        sub = grid
    else:
        rob = pd.read_csv(P['subsets'] / 'robustness.csv', dtype={'video_id': str})
        if n <= len(rob):
            keep = set(stratified_subset(rob, n, CFG['seed'] + 5).uid)
        else:
            rest = grid[~grid.uid.isin(set(rob.uid))]
            keep = set(rob.uid) | set(stratified_subset(rest, n - len(rob), CFG['seed'] + 5).uid)
        sub = grid[grid.uid.isin(keep)].reset_index(drop=True)
    if SHARD == 0 and ST_NOTEBOOK == '02_cv_detect_track':   # written once, by 02 only
        sub.to_csv(path, index=False)
        log(f'main_grid_eval.csv frozen: {len(sub)} questions, {sub.video_id.nunique()} videos')
    return sub

def main_grid_eval_hash(sub):
    return hashlib.sha1(sub.uid.astype(str).str.cat(sep='|').encode()).hexdigest()[:12]

_ZIP_INDEX = {}
def zip_index(name):
    """{video stem: member path} for a video zip on Drive (cached)."""
    if name not in _ZIP_INDEX:
        zp = P['data'] / name
        if not zp.exists():
            _ZIP_INDEX[name] = {}
        else:
            with zipfile.ZipFile(zp) as zf:
                _ZIP_INDEX[name] = {Path(n).stem: n for n in zf.namelist() if n.lower().endswith('.mp4')}
    return _ZIP_INDEX[name]

GROUP_ORDER = {'dev': 0, 'star': 1, 'main': 2, 'intentqa': 3}
def build_work_list():
    """Every video the CV pipeline must process, with its source zip, duration and shard.
    Shards are balanced by total video seconds, tier by tier, and are the same on every
    member's machine (deterministic), so nobody needs to coordinate."""
    meta = pd.read_csv(P['ann'] / 'video_meta.csv', dtype={'video_id': str}).set_index('video_id').duration_s
    rows, missing = [], {}
    nextqa_sets = [('dev', pd.read_csv(P['subsets'] / 'dev.csv', dtype={'video_id': str}), 'videos_nextqa_eval.zip', 1),
                   ('main', main_grid_eval(), 'videos_nextqa_eval.zip', 1),
                   ('intentqa', pd.read_csv(P['subsets'] / 'intentqa_eval.csv', dtype={'video_id': str}),
                    'videos_intentqa_eval.zip', 2)]
    for group, df, zname, tier in nextqa_sets:
        idx = zip_index(zname)
        for v in sorted(df.video_id.unique()):
            if v not in idx:
                missing.setdefault(group, []).append(v); continue
            rows.append({'key': v, 'group': group, 'tier': tier, 'zip': zname, 'member': idx[v],
                         'duration_s': meta.get(v, np.nan)})
    man = P['data'] / 'star' / 'clip_manifest.csv'
    if man.exists():
        M = pd.read_csv(man)
        idx = zip_index('videos_star_eval.zip')
        for r in M[M.version == 'orig'].itertuples(index=False):
            if r.clip_id not in idx:
                missing.setdefault('star', []).append(r.clip_id); continue
            rows.append({'key': r.clip_id, 'group': 'star', 'tier': 1, 'zip': 'videos_star_eval.zip',
                         'member': idx[r.clip_id], 'duration_s': r.n_frames_src / r.fps})
    else:
        log('data/star/clip_manifest.csv not found: run 01b_star first. STAR clips are skipped for now.', 'WARN')
    W = pd.DataFrame(rows).drop_duplicates('key').reset_index(drop=True)
    W['duration_s'] = W.duration_s.fillna(W.duration_s.median()).astype(float)
    W['shard'] = -1
    for tier in sorted(W.tier.unique()):              # greedy: longest video to the least-loaded shard
        load = [0.0] * N_SHARDS
        sub = W[W.tier == tier].sort_values(['duration_s', 'key'], ascending=[False, True])
        for i, d in zip(sub.index, sub.duration_s):
            s = int(np.argmin(load)); W.at[i, 'shard'] = s; load[s] += d
    W['order'] = W.group.map(GROUP_ORDER)
    W = W.sort_values(['tier', 'order', 'key']).drop(columns='order').reset_index(drop=True)
    for g, v in missing.items():
        log(f'{len(v)} {g} videos are not in their zip and are skipped: {v[:5]}', 'WARN')
    return W

class VideoSource:
    """Extracts one video at a time from its Drive zip to the Colab disk, and deletes it after use."""
    def __init__(self):
        self.dir = P['local_videos'] / 'cv'
        self.dir.mkdir(parents=True, exist_ok=True)
        self.zf = {}
    def get(self, row):
        out = self.dir / f'{row.key}.mp4'
        if not out.exists():
            if row.zip not in self.zf:
                self.zf[row.zip] = zipfile.ZipFile(P['data'] / row.zip)
            part = out.with_suffix('.part')
            with self.zf[row.zip].open(row.member) as s, open(part, 'wb') as d:
                shutil.copyfileobj(s, d)
            os.replace(part, out)
        return out
    def drop(self, row):
        (self.dir / f'{row.key}.mp4').unlink(missing_ok=True)

def select_work(W, groups, limit=None):
    """This member's videos, in processing order (Tier 1 first)."""
    mine = W[(W.shard == SHARD) & W.group.isin(groups)]
    if limit:
        mine = mine.groupby('group', sort=False).head(limit)
    mine = mine.assign(_o=mine.group.map({g: i for i, g in enumerate(groups)}))
    return mine.sort_values(['tier', '_o', 'key']).drop(columns='_o').reset_index(drop=True)

def delete_model_cache(*repo_ids):
    """Free Colab disk: remove downloaded Hugging Face weights that are no longer needed."""
    hub = Path(os.environ.get('HF_HOME', Path.home() / '.cache' / 'huggingface')) / 'hub'
    for rid in repo_ids:
        d = hub / ('models--' + rid.replace('/', '--'))
        if d.exists():
            shutil.rmtree(d, ignore_errors=True)
            log(f'deleted cached weights: {rid}')

WORK = build_work_list()
MGE_HASH = main_grid_eval_hash(main_grid_eval())
print('main_grid_eval hash:', MGE_HASH)
summary = WORK.pivot_table(index='group', columns='shard', values='duration_s', aggfunc=['count', 'sum'])
summary.columns = [f'{a}_shard{b}' for a, b in summary.columns]
print(summary.round(0).astype(int).to_string())

[12:41:27] INFO  restored 0 files from 105 packs on Drive
main_grid_eval hash: e83b2604bc79
          count_shard0  sum_shard0
group                             
dev                 92        3880
intentqa           224       10612
main               175        6898
star               360        4753


In [13]:
EV_DIR, CAP_DIR = LOCAL_CV / 'events', LOCAL_CV / 'captions'
STAR = P['data'] / 'star'
MAN = pd.read_csv(STAR / 'clip_manifest.csv') if (STAR / 'clip_manifest.csv').exists() else pd.DataFrame()
TIER1 = WORK[WORK.group.isin(GROUPS)]
have = done_ids(EV_DIR)
missing = TIER1[~TIER1.key.isin(have)]
print(f'videos with 03 results: {len(set(TIER1.key) & have)} of {len(TIER1)}')
print(missing.groupby('group').size().rename('missing').to_string() if len(missing) else 'none missing')
if len(missing[missing.group != 'intentqa']):
    log(f'{len(missing[missing.group != "intentqa"])} Tier 1 videos have no 03 results yet: their memories are skipped '
        'for now. Re-run this notebook after notebook 03 is finished.', 'WARN')

videos with 03 results: 630 of 851
group
intentqa    221


## 2. STAR pause and sham versions

For each `star_gap` question, four versions (pause 10 s, sham 10 s, pause 25 s, sham 25 s) are made from the original clip's events, states and captions. Every time at or after the insertion point moves later by the pause length; an event or state interval that spans the insertion point is lengthened by it. The same shift is applied to the oracle ground truth.

In [14]:
def shift_fns(at, secs):
    s0 = lambda t: round(t + secs, 2) if t >= at - 1e-9 else t        # starts at/after the insert move
    s1 = lambda t: round(t + secs, 2) if t > at + 1e-9 else t         # ends after the insert move
    return s0, s1

def shift_events(ev, at, secs):
    ev = json.loads(json.dumps(ev))
    s0, s1 = shift_fns(at, secs)
    for e in ev['events']:
        e['t0'], e['t0_win'] = s0(e['t0']), s0(e['t0_win'])
        e['t1'], e['t1_win'] = s1(e['t1']), s1(e['t1_win'])
    for c in ev['changes']:
        c['t'] = s0(c['t'])
    for o in ev['objects'].values():
        for iv in o['intervals']:
            iv['t0'], iv['t1'] = s0(iv['t0']), s1(iv['t1'])
    ev['duration_s'] = round(ev['duration_s'] + secs, 3)
    return ev

def shift_captions(cap, at, secs):
    cap = json.loads(json.dumps(cap))
    s0, s1 = shift_fns(at, secs)
    for c in cap['captions']:
        c['ws'], c['we'] = s0(c['ws']), s1(c['we'])
    cap['duration_s'] = round(cap['duration_s'] + secs, 3)
    return cap

def shift_oracle(orc, at, secs):
    orc = json.loads(json.dumps(orc))
    s0, s1 = shift_fns(at, secs)
    for e in orc['events']:
        e['start'], e['end'] = s0(e['start']), s1(e['end'])
    for k in orc['keyframes']:
        k['t'] = s0(k['t'])
    orc['duration_s'] = round(orc['duration_s'] + secs, 2)
    return orc

# every clip key (NExT-QA video or STAR clip version) -> where its inputs come from
SOURCES = {}
for k in sorted(set(TIER1.key) & have):
    SOURCES[k] = {'src': k, 'at': None, 'secs': 0.0}
n_gap, n_gap_missing = 0, 0
# pause/sham positions come from star_gap.csv (01b), so this does not depend on the clip manifest
GAPSET = pd.read_csv(P['subsets'] / 'star_gap.csv')
PAUSES = {'10': float(CFG['star'].get('pause_medium_s', 10.0)), '25': float(CFG['star'].get('pause_long_s', 25.0))}
for r in GAPSET.itertuples(index=False):
    src = f'{r.uid}__orig'
    for tag, secs in PAUSES.items():
        for kind, at in [('pause', r.pause_at_s), ('sham', r.sham_at_s)]:
            if src in have:
                SOURCES[f'{r.uid}__{kind}{tag}'] = {'src': src, 'at': float(at), 'secs': secs}; n_gap += 1
            else:
                n_gap_missing += 1
print(f'pause/sham versions: {n_gap} ready, {n_gap_missing} waiting for their original clip')

pause/sham versions: 480 ready, 0 waiting for their original clip


## 3. One fact set, many formats

Events and states are first put into one common structure, so every format renders the same facts:

- **Events:** start and end (snapped where possible), who, action, object, place. Lines the model wrote about things merely appearing or disappearing are dropped. Duplicate lines are merged.
- **Objects:** state intervals (`on table_2`, `held by person_1`, `not visible`, …) for every object that takes part in an event or has a confirmed state change; scene parts and worn items are left out. A state that the VLM did not confirm is marked *uncertain*.
- **State changes:** where one interval ends and the next begins.

The oracle fact set uses the same structure: Charades actions are the events, and STAR's keyframe relationships become the object states.

In [15]:
NOISE_VERBS = {'appears', 'appear', 'disappears', 'disappear', 'is visible', 'visible', 'is seen', 'seen',
               'is present', 'present', 'exists', 'is shown', 'shown', 'comes into view', 'leaves view'}
WORN = {'glasses', 'sunglasses', 'hat', 'cap', 'helmet', 'shoe', 'shoes', 'sock', 'socks', 'shirt', 'jacket', 'coat',
        'dress', 'scarf', 'glove', 'gloves', 'ring', 'necklace', 'bracelet', 'watch', 'tie', 'mask', 'costume',
        'diaper', 'bib', 'pants', 'shorts', 'skirt', 'boot', 'boots'}
MAX_INTERVALS = 8

def state_phrase(state, target):
    return {'held_by': f'held by {target}', 'on': f'on {target}', 'near': f'near {target}',
            'absent': 'not visible', 'visible': 'visible'}.get(state, state)

def facts_from_cv(ev, cap):
    events, seen = [], set()
    for e in ev['events']:
        verb = (e.get('verb') or '').strip().lower()
        if not verb or verb in NOISE_VERBS:
            continue
        who = e.get('actor') or e.get('actor_text') or 'someone'
        obj = e.get('object') or e.get('object_text')
        loc = e.get('location')
        if loc and loc.strip().lower() in ('none', '-', ''):
            loc = None
        key = (round(e['t0'], 1), round(e['t1'], 1), who, verb, obj)
        if key in seen:
            continue
        seen.add(key)
        events.append({'t0': float(e['t0']), 't1': float(e['t1']), 'who': who, 'verb': verb, 'obj': obj, 'loc': loc})
    events.sort(key=lambda x: (x['t0'], x['t1']))
    used = {x['who'] for x in events} | {x['obj'] for x in events if x['obj']}
    confirmed = {c['obj'] for c in ev['changes'] if c.get('status') == 'confirmed'}
    objects, changes = {}, []
    for oid, o in ev['objects'].items():
        if o.get('scene') or o['cls'] in WORN or (oid not in used and oid not in confirmed):
            continue
        ivs = []
        for iv in o['intervals']:
            ivs.append({'t0': iv['t0'], 't1': iv['t1'], 'text': state_phrase(iv['state'], iv['target']),
                        'status': 'uncertain' if iv.get('status') == 'uncertain' else 'ok'})
        objects[oid] = {'cls': o['cls'], 'intervals': ivs}
        for prev, cur in zip(ivs, ivs[1:]):
            changes.append({'t': cur['t0'], 'obj': oid, 'from': prev['text'], 'to': cur['text'], 'status': cur['status']})
    changes.sort(key=lambda c: (c['t'], c['obj']))
    captions = [{'t0': c['ws'], 't1': c['we'], 'text': c['text']} for c in (cap or {}).get('captions', []) if c.get('text')]
    return {'duration': float(ev['duration_s']), 'events': events, 'objects': objects, 'changes': changes,
            'captions': captions, 'source': 'cv'}

def facts_from_oracle(orc):
    events = [{'t0': float(e['start']), 't1': float(e['end']), 'who': 'person',
               'verb': e['event'][0].lower() + e['event'][1:], 'obj': None, 'loc': None} for e in orc['events']]
    events.sort(key=lambda x: (x['t0'], x['t1']))
    # relationship triples -> intervals over consecutive keyframes
    runs = {}
    kfs = sorted(orc['keyframes'], key=lambda k: k['t'])
    for i, k in enumerate(kfs):
        for a, rel, b in k['relations']:
            key = (b, f'{a} {rel.replace("_", " ")} {b}')
            r = runs.setdefault(key, [])
            if r and r[-1][2] == i - 1:
                r[-1][1], r[-1][2] = k['t'], i
            else:
                r.append([k['t'], k['t'], i])
    objects, changes = {}, []
    for (b, text), rr in runs.items():
        o = objects.setdefault(b, {'cls': b, 'intervals': []})
        for t0, t1, _ in rr:
            o['intervals'].append({'t0': t0, 't1': t1, 'text': text, 'status': 'ok'})
    for b, o in objects.items():
        o['intervals'].sort(key=lambda x: (x['t0'], x['text']))
        for iv in o['intervals']:
            changes.append({'t': iv['t0'], 'obj': b, 'from': None, 'to': iv['text'], 'status': 'ok'})
    changes.sort(key=lambda c: (c['t'], c['obj']))
    return {'duration': float(orc['duration_s']), 'events': events, 'objects': objects, 'changes': changes,
            'captions': [], 'source': 'oracle'}

## 4. The renderers

Each memory is a list of **items** (one line each, with its times), so strategies C and D in notebook 05 can retrieve items under the same token budget for every format. The full text is the items joined, under short section headers.

- **M1** turns IDs into words (`person_1` → "person 1") and says "possibly" where a state change is uncertain.
- **M4** relations are computed, never generated: Allen relations between each event and the next one, DURING when one event lies inside another, and CO_OCCURS when a state change of an object starts within 1 s of an event involving that object. CO_OCCURS is a candidate link, not a claim of cause.

In [16]:
TOL = CFG['memory'].get('overlap_tolerance_s', 0.5)
COOC = CFG['memory'].get('cooccur_window_s', 1.0)

def words_for_ids(text):
    return re.sub(r'\b([a-z]+(?:_[a-z]+)*?)_(?:ov)?(\d+)\b', lambda m: m.group(1).replace('_', ' ') + ' ' + m.group(2), text)

def tfmt(a, b=None):
    return f'{a:.1f}s' if b is None else f'{a:.1f}-{b:.1f}s'

def ev_text(e):
    s = f"{e['who']} {e['verb']}" + (f" {e['obj']}" if e['obj'] else '')
    return s + (f" ({e['loc']})" if e['loc'] else '')

def items_events(F):
    return [{'id': f'E{i}', 'kind': 'event', 't0': e['t0'], 't1': e['t1'],
             'text': f"E{i} [t={tfmt(e['t0'], e['t1'])}] {ev_text(e)}"} for i, e in enumerate(F['events'], 1)]

def items_states(F):
    out = []
    for oid, o in sorted(F['objects'].items(), key=lambda kv: (kv[1]['intervals'][0]['t0'] if kv[1]['intervals'] else 0, kv[0])):
        ivs = o['intervals']
        if not ivs or all(iv['text'] in ('visible', 'not visible') for iv in ivs) and F['source'] == 'cv' and len(ivs) <= 1:
            continue
        parts = [f"{tfmt(iv['t0'], iv['t1'])} {iv['text']}" + (' (uncertain)' if iv['status'] == 'uncertain' else '')
                 for iv in ivs[:MAX_INTERVALS]]
        if len(ivs) > MAX_INTERVALS:
            parts.append('...')
        label = oid if F['source'] == 'oracle' or o['cls'].replace(' ', '_') in oid else f"{oid} ({o['cls']})"
        out.append({'id': f'S_{oid}', 'kind': 'state', 't0': ivs[0]['t0'], 't1': ivs[-1]['t1'],
                    'text': f'{label} | ' + ' | '.join(parts)})
    return out

def render_M1(F):
    sents = []
    for o_id, o in F['objects'].items():
        if o['intervals'] and F['source'] == 'cv':
            iv = o['intervals'][0]
            if iv['t0'] <= 0.5 and iv['text'] != 'not visible':
                sents.append((0.0, 0, f"At the start, {o_id} is {iv['text']}."))
    for e in F['events']:
        tail = f" {e['obj']}" if e['obj'] else ''
        place = f" at {e['loc']}" if e['loc'] else ''
        sents.append((e['t0'], 1, f"From {e['t0']:.1f} s to {e['t1']:.1f} s, {e['who']} {e['verb']}{tail}{place}."))
    for c in F['changes']:
        if F['source'] == 'oracle':
            sents.append((c['t'], 2, f"At {c['t']:.1f} s: {c['to']}."))
        else:
            verb = 'may now be' if c['status'] == 'uncertain' else 'is now'
            sents.append((c['t'], 2, f"At {c['t']:.1f} s, {c['obj']} {verb} {c['to']}."))
    sents.sort(key=lambda x: (x[0], x[1]))
    items = [{'id': f'P{i}', 'kind': 'sentence', 't0': t, 't1': t, 'text': words_for_ids(s)}
             for i, (t, _, s) in enumerate(sents, 1)]
    return items, [('', items)]

def render_M2(F):
    ev = items_events(F)
    return ev, [('EVENTS', ev)]

def render_M3(F):
    ev, st = items_events(F), items_states(F)
    return ev + st, [('EVENTS', ev), ('OBJECT STATES', st)]

def allen(a, b):
    if a[1] < b[0] - TOL:
        return 'BEFORE'
    if b[1] < a[0] - TOL:
        return 'AFTER'
    if a[0] >= b[0] - TOL and a[1] <= b[1] + TOL and (a[1] - a[0]) < (b[1] - b[0]) - TOL:
        return 'DURING'
    return 'OVERLAPS'

def summary_item(F):
    people = sorted({e['who'] for e in F['events'] if str(e['who']).startswith('person')})
    from collections import Counter
    verbs = Counter(e['verb'] for e in F['events']).most_common(4)
    objs = Counter(e['obj'] for e in F['events'] if e['obj']).most_common(4)
    s = (f"Video of {F['duration']:.0f} s with {len(F['events'])} events and {len(F['changes'])} state changes. "
         f"People: {', '.join(people) or 'none detected'}. "
         f"Main objects: {', '.join(o for o, _ in objs) or 'none'}. "
         f"Most frequent actions: {', '.join(f'{v} ({n})' for v, n in verbs) or 'none'}.")
    return {'id': 'SUM', 'kind': 'summary', 't0': 0.0, 't1': F['duration'], 'text': s}

def render_M4(F):
    ev, st = items_events(F), items_states(F)
    segs, seg_len = [], 10.0
    n_seg = max(1, int(np.ceil(F['duration'] / seg_len)))
    for k in range(n_seg):
        a, b = k * seg_len, min((k + 1) * seg_len, F['duration'])
        ids = [x['id'] for x in ev if a <= x['t0'] < b]
        who = sorted({e['who'] for e, x in zip(F['events'], ev) if x['id'] in ids})
        segs.append({'id': f'SEG{k + 1}', 'kind': 'segment', 't0': a, 't1': b,
                     'text': f"SEG{k + 1} [t={tfmt(a, b)}] events {', '.join(ids) if ids else 'none'}"
                             + (f"; who: {', '.join(who)}" if who else '')})
    rel = []
    E = F['events']
    for i in range(len(E) - 1):
        a, b = (E[i]['t0'], E[i]['t1']), (E[i + 1]['t0'], E[i + 1]['t1'])
        rel.append({'id': f'R{len(rel) + 1}', 'kind': 'relation', 't0': min(a[0], b[0]), 't1': max(a[1], b[1]),
                    'text': f'E{i + 1} {allen(a, b)} E{i + 2}'})
    for i, x in enumerate(E):
        for j, y in enumerate(E):
            if i != j and j != i + 1 and i != j + 1 and allen((x['t0'], x['t1']), (y['t0'], y['t1'])) == 'DURING':
                rel.append({'id': f'R{len(rel) + 1}', 'kind': 'relation', 't0': y['t0'], 't1': y['t1'],
                            'text': f'E{i + 1} DURING E{j + 1}'})
    for c in F['changes']:
        for i, e in enumerate(E, 1):
            if c['obj'] in (e['who'], e['obj']) and abs(c['t'] - e['t0']) <= COOC:
                what = f"{c['obj']} becomes {c['to']}" if F['source'] == 'cv' else c['to']
                rel.append({'id': f'R{len(rel) + 1}', 'kind': 'relation', 't0': c['t'], 't1': e['t1'],
                            'text': f"[t={tfmt(c['t'])}] {what} CO_OCCURS E{i}"})
    summ = summary_item(F)
    return [summ] + segs + ev + st + rel, [('SUMMARY', [summ]), ('SEGMENTS', segs), ('EVENTS', ev),
                                           ('OBJECT STATES', st), ('RELATIONS', rel)]

def render_CAP(F):
    it = [{'id': f'C{i}', 'kind': 'caption', 't0': c['t0'], 't1': c['t1'], 'text': f"[t={tfmt(c['t0'], c['t1'])}] {c['text']}"}
          for i, c in enumerate(F['captions'], 1)]
    return it, [('CAPTIONS', it)]

def render_VA(F):
    caps, _ = render_CAP(F)
    table = []
    for oid, o in sorted(F['objects'].items()):
        seen = [iv for iv in o['intervals'] if iv['text'] != 'not visible']
        if not seen:
            continue
        states = sorted({iv['text'] for iv in seen if iv['text'] != 'visible'})
        table.append({'id': f'O_{oid}', 'kind': 'object', 't0': seen[0]['t0'], 't1': seen[-1]['t1'],
                      'text': f"{oid} | {o['cls']} | seen {tfmt(seen[0]['t0'], seen[-1]['t1'])} | "
                              f"states: {', '.join(states) if states else 'none'}"})
    for p in sorted({e['who'] for e in F['events'] if str(e['who']).startswith('person')}):
        ts = [(e['t0'], e['t1']) for e in F['events'] if e['who'] == p]
        table.append({'id': f'O_{p}', 'kind': 'object', 't0': min(a for a, _ in ts), 't1': max(b for _, b in ts),
                      'text': f"{p} | person | active {tfmt(min(a for a, _ in ts), max(b for _, b in ts))}"})
    return caps + table, [('SEGMENT CAPTIONS', caps), ('OBJECT TABLE', table)]

RENDER = {'M1': render_M1, 'M2': render_M2, 'M3': render_M3, 'M4': render_M4, 'CAP': render_CAP, 'VA': render_VA}

def to_text(sections):
    blocks = []
    for head, items in sections:
        lines = [x['text'] for x in items]
        if not lines:
            continue
        blocks.append((f'{head}:\n' if head else '') + ('\n'.join(lines) if head else ' '.join(lines)))
    return '\n\n'.join(blocks) if blocks else '(nothing detected)'

## 5. Build every memory

Real memories (M1–M4, CAP, VA) for every NExT-QA video and STAR clip version that has 03 results; oracle memories (ORACLE_M1–M4) for every STAR clip version from the ground truth saved by `01b_star`. Tokens are counted with the main reasoner's tokenizer (Qwen3).

In [17]:
TOK = None
if not MOCK:
    try:
        from transformers import AutoTokenizer
        spec = CFG['models']['reasoner_main']
        TOK = AutoTokenizer.from_pretrained(spec['id'], revision=spec.get('revision', 'main'))
        TOK_NAME = spec['id']
    except Exception as e:
        log(f'tokenizer not available ({e.__class__.__name__}): token counts are estimated from words', 'WARN')
def n_tokens(text):
    return len(TOK(text)['input_ids']) if TOK is not None else int(round(len(text.split()) * 1.35))

LOCAL_MEM = P['local'] / 'memory_out'
shutil.rmtree(LOCAL_MEM, ignore_errors=True)
LOCAL_MEM.mkdir(parents=True)
ORACLE_DIR = STAR / 'oracle'
uid_of = lambda key: key.split('__')[0]

def write_memory(fmt, key, F, items, sections, meta):
    text = to_text(sections)
    rec = {'key': key, 'format': fmt, 'text': text, 'items': items, 'n_items': len(items),
           'words': len(text.split()), 'tokens': n_tokens(text), **meta}
    out = LOCAL_MEM / fmt / f'{key}.json'
    out.parent.mkdir(parents=True, exist_ok=True)
    save_compact(rec, out)
    return {'key': key, 'format': fmt, 'n_items': rec['n_items'], 'words': rec['words'], 'tokens': rec['tokens'],
            'events': len(F['events']), 'objects': len(F['objects']), 'duration_s': F['duration'], **meta}

STATS, problems = [], []
t0 = time.time()
for i, (key, s) in enumerate(sorted(SOURCES.items()), 1):
    try:
        ev = load_json(EV_DIR / f"{s['src']}.json")
        cap = load_json(CAP_DIR / f"{s['src']}.json", default={'captions': [], 'duration_s': ev['duration_s']})
        if s['at'] is not None:
            ev, cap = shift_events(ev, s['at'], s['secs']), shift_captions(cap, s['at'], s['secs'])
        F = facts_from_cv(ev, cap)
        meta = {'group': WORK.set_index('key').group.get(s['src'], 'star'), 'source': 'cv'}
        for fmt, fn in RENDER.items():
            items, sections = fn(F)
            STATS.append(write_memory(fmt, key, F, items, sections, meta))
        if key.startswith('star_') and (ORACLE_DIR / f'{uid_of(key)}.json').exists():
            orc = load_json(ORACLE_DIR / f'{uid_of(key)}.json')
            if s['at'] is not None:
                orc = shift_oracle(orc, s['at'], s['secs'])
            G = facts_from_oracle(orc)
            for fmt in ['M1', 'M2', 'M3', 'M4']:
                items, sections = RENDER[fmt](G)
                STATS.append(write_memory(f'ORACLE_{fmt}', key, G, items, sections, {'group': 'star', 'source': 'oracle'}))
    except Exception as e:
        problems.append((key, f'{e.__class__.__name__}: {e}'))
    if i % 200 == 0:
        log(f'memories: {i}/{len(SOURCES)} clips ({time.time() - t0:.0f} s)')
S = pd.DataFrame(STATS)
print(f'built {len(S)} memories for {S.key.nunique()} clips in {time.time() - t0:.0f} s; problems: {len(problems)}')
for k, p in problems[:10]:
    print('  ', k, p)

[12:42:01] INFO  memories: 200/1110 clips (27 s)
[12:42:29] INFO  memories: 400/1110 clips (56 s)
[12:42:47] INFO  memories: 600/1110 clips (73 s)
[12:43:07] INFO  memories: 800/1110 clips (93 s)
[12:43:27] INFO  memories: 1000/1110 clips (113 s)
built 10020 memories for 1110 clips in 124 s; problems: 0


## 6. Upload to Drive (one verified zip) and check

All memories go into one zip in `outputs/memory/`, which is read back from Drive to prove it arrived. The table shows the median size of each format; M4 is expected to be the largest, which is why strategies C and D give every format the same retrieval budget.

In [18]:
stamp = dt.datetime.now().strftime('%Y%m%d-%H%M%S')
name = f'04_memories_{stamp}.zip'
tmp = P['local_tmp'] / name
with zipfile.ZipFile(tmp, 'w', zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    for f in sorted(LOCAL_MEM.rglob('*.json')):
        zf.write(f, str(f.relative_to(LOCAL_MEM)))
n_files = len(list(LOCAL_MEM.rglob('*.json')))
dest = P['memory'] / name
ok = False
for attempt in range(3):
    try:
        shutil.copyfile(tmp, dest)
        with zipfile.ZipFile(dest) as zf:
            ok = len(zf.namelist()) == n_files and zf.testzip() is None
        if ok:
            break
    except Exception as e:
        log(f'upload failed ({e.__class__.__name__}: {e}); retrying', 'WARN')
    time.sleep(5 * (attempt + 1))
if not ok:
    raise RuntimeError(f'Drive did not keep {name}. Do not restart: run this cell again.')
log(f'uploaded {name}: {n_files} memories ({_gb(dest.stat().st_size) * 1024:.1f} MB), verified on Drive')

S.to_csv(P['memory'] / 'memory_stats.csv', index=False)
summary = (S.groupby('format').agg(clips=('key', 'nunique'), median_items=('n_items', 'median'),
                                   median_tokens=('tokens', 'median'), p90_tokens=('tokens', lambda x: x.quantile(0.9)),
                                   max_tokens=('tokens', 'max'))
           .reindex(['M1', 'M2', 'M3', 'M4', 'CAP', 'VA', 'ORACLE_M1', 'ORACLE_M2', 'ORACLE_M3', 'ORACLE_M4']).dropna(how='all'))
print(summary.round(0).astype(int).to_string())

# one example video, every format, to read
ex_key = next((k for k in sorted(SOURCES) if k.endswith('__orig') and (LOCAL_MEM / 'ORACLE_M4' / f'{k}.json').exists()),
              next(iter(sorted(SOURCES)), None))
if ex_key:
    lines = []
    for fmt in ['M1', 'M2', 'M3', 'M4', 'CAP', 'VA', 'ORACLE_M2', 'ORACLE_M4']:
        f = LOCAL_MEM / fmt / f'{ex_key}.json'
        if f.exists():
            r = load_json(f)
            lines.append(f'===== {fmt} ({r["tokens"]} tokens) =====\n{r["text"]}\n')
    (P['memory'] / 'examples.txt').write_text(f'example clip: {ex_key}\n\n' + '\n'.join(lines))
    print(f'\nexample clip {ex_key}:\n')
    print('\n'.join(l[:1500] for l in lines[:4]))
RUN = Run(params={'step': 'memory_build', 'clips': int(S.key.nunique()), 'pack': name})
RUN.finish('done' if not problems else 'partial', notes=f'{len(S)} memories; {len(problems)} problems')

[12:43:42] INFO  uploaded 04_memories_20261017-124338.zip: 10020 memories (10.2 MB), verified on Drive
           clips  median_items  median_tokens  p90_tokens  max_tokens
format                                                               
M1          1110            30            607        1693        8122
M2          1110            14            372         928        3081
M3          1110            20            674        1818        8136
M4          1110            44           1108        2908       12023
CAP         1110             6            127         343         726
VA          1110            13            308         755        2802
ORACLE_M1    840            27            402         737        1769
ORACLE_M2    840             7            156         251         411
ORACLE_M3    840            10            431         682        1048
ORACLE_M4    840            28            690        1080        2204

example clip star_Interaction_T1_1108__orig:

===== M1 (

## What to send back

The output of section 6: the size table and the example. Notebook 05 (the LLM runs) is sized from the token counts.